In [14]:
import os 
import time 
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [15]:
save_folder = 'run8'

n = 5

lower_factor = 0.99
upper_factor = 2 - lower_factor

b_max = 30
q_max = 0.2



In [16]:
# Load experimental data
atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)
x_totem, y_totem, yerr_totem = process_data(totem_data, totem_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]


/tmp/ipykernel_12802/2977988099.py:2: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
/tmp/ipykernel_12802/2977988099.py:3: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  totem_data = pd.read_csv('../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)


In [17]:

b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'eps': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'eps': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    }
}


# Get parameters for selected configuration
initial_params_log_atlas = ensemble_parameters['atlas']['log']
initial_params_pl_atlas = ensemble_parameters['atlas']['pl']



In [18]:
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def born_amp(diff_T, s, epsilon, t):
    
    alpha_pomeron = 1.0 + epsilon + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T


In [19]:
# Trapezoid rule (scalar evaluation only)
def trapezoid(func, a, b, n):
    dx = (b - a) / n
    total = 0.0
    x_prev = a
    y_prev = func(x_prev)
    for i in range(1, n+1):
        x_curr = a + i * dx
        y_curr = func(x_curr)
        total += (y_prev + y_curr) * dx / 2
        x_prev, y_prev = x_curr, y_curr
    return total

# Adaptive trapezoid (scalar version)
def adaptive_trapezoid(func, a, b, tol=1e-8):
    if a == b:
        return 0.0
    n = 2
    prev = trapezoid(func, a, b, n)
    while True:
        n *= 2
        curr = trapezoid(func, a, b, n)
        if abs(curr - prev) < tol:
            return curr
        prev = curr

# Complex version
def adaptive_trapezoid_complex(func, a, b, tol=1e-8):
    real_part = adaptive_trapezoid(lambda x: np.real(func(x)), a, b, tol)
    imag_part = adaptive_trapezoid(lambda x: np.imag(func(x)), a, b, tol)
    return real_part + 1j * imag_part

In [20]:
from numpy.polynomial.legendre import leggauss

# -------------------------------
# Vectorized integral over phi
# -------------------------------
def phi_integral_vec(k, mg, a1, a2, m2_func, q, n=50):
    # Get nodes and weights for phi integration
    x, w = leggauss(n)
    # Map [-1,1] -> [0, 2pi]
    phi = 0.5 * (x + 1) * 2*np.pi
    w = w * np.pi  # adjust weights

    # Broadcast k and phi for vectorized evaluation
    k = np.atleast_1d(k)[:, None]  # shape (Nk,1)
    phi = phi[None, :]             # shape (1,Nphi)

    integrand = k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                     T_2(k, phi, mg, a1, a2, m2_func, q))

    # Sum over phi dimension
    return np.sum(integrand * w, axis=1)

# -------------------------------
# Vectorized integral over k
# -------------------------------
def compute_k_phi_integral_vec(sqrt_s_val, mg, a1, a2, m2_func, q, n=50):
    # Get nodes and weights for k integration
    x, w = leggauss(n)
    # Map [-1,1] -> [0, sqrt_s_val]
    k = 0.5 * (x + 1) * sqrt_s_val
    w = w * 0.5 * sqrt_s_val

    # Call vectorized phi integral for all k at once
    integrand_vals = phi_integral_vec(k, mg, a1, a2, m2_func, q, n=n)

    # Weighted sum over k
    return np.sum(integrand_vals * w)

In [21]:
import numpy as np
from numpy.polynomial.legendre import leggauss
from scipy.special import j0

# -------------------------------
# Chi integral, over q
# -------------------------------
def chi_integral_vec(sqrt_s_val, b, q_max, eps, mg, a1, a2, m2_func, n=80):
    s = float(sqrt_s_val**2)

    # Gauss–Legendre nodes/weights para integração em q
    x, w = leggauss(n)
    q = 0.5 * (x + 1) * q_max
    w = w * 0.5 * q_max

    # Recalcular a amplitude de Born em cada ponto q
    born_vals = []
    for q_val in q:
        t = -q_val**2
        diff_T = compute_k_phi_integral_vec(
            sqrt_s_val=sqrt_s_val,
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q_val,
            n=n
        )
        born_vals.append(born_amp(diff_T, s, eps, t))
    born_vals = np.array(born_vals)

    # Integrando vetorizado
    integrand = (q * j0(b * q) * born_vals) / s

    return np.sum(integrand * w)


# -------------------------------
# Eikonal amplitude integral, over b
# -------------------------------
def eik_amp_vec(sqrt_s_values, b_max, q_max, eps, mg, a1, a2, m2_func, n=80):
    if isinstance(sqrt_s_values, (int, float, complex)):
        sqrt_s_values = [sqrt_s_values]

    amp_list = []

    # Gauss–Legendre nodes/weights for b integration
    xb, wb = leggauss(n)
    b_nodes = 0.5 * (xb + 1) * b_max
    wb = wb * 0.5 * b_max

    for sqrt_s_val in sqrt_s_values:
        s = float(sqrt_s_val**2)

        # Compute chi integral for all b
        chi_vals = np.array([
            chi_integral_vec(
                sqrt_s_val,
                b,
                q_max,
                eps,
                mg,
                a1,
                a2,
                m2_func,
                n=n
            )
            for b in b_nodes
        ])

        # Eikonal amplitude integrand
        integrand = b_nodes * (1 - np.exp(1j * chi_vals))

        # Integrate over b
        integral_result = np.sum(integrand * wb)
        A_eik = 1j * s * integral_result
        amp_list.append(A_eik)

    return amp_list if len(amp_list) > 1 else amp_list[0]

In [22]:
def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323

In [23]:
import numpy as np

# Lista de valores de q para a integral eikonal
lst_q_integration = np.linspace(0, 0.1, 10)

def model_function(x_born, eps, mg, a1, a2, sqrt_s, model_type='log', n=50):
    """
    x_born: array com os valores experimentais de q^2
    eps, mg, a1, a2: parâmetros do modelo
    sqrt_s: energia
    model_type: 'pl' ou 'log'
    """
    m2_func = m2_log if model_type == 'log' else m2_pl
    s = sqrt_s ** 2

    # -------------------------------
    # Vetoriza integral sobre q
    # -------------------------------
    # Chamamos compute_k_phi_integral para todos q_int de uma vez
    # (precisa que compute_k_phi_integral_vec seja usado em vez do antigo)
    T_vals = np.array([
        compute_k_phi_integral_vec(sqrt_s, mg, a1, a2, m2_func, q, n=n)
        for q in lst_q_integration
    ])

    # Integral sobre q (mesma para todos q_exp, se integrando não depende de x_born)
    diff_T = np.trapezoid(T_vals, lst_q_integration)

    # -------------------------------
    # Vetoriza sobre pontos experimentais
    # -------------------------------
    # born_amp deve ser vetorizado sobre array de t (= -x_born ou função dele)
    t_vals = -np.array(x_born)  # supondo que t = -q^2_exp
    born_amplitudes = born_amp(diff_T, s, eps, t_vals)

    # eik_amp deve aceitar arrays de born_amplitude → já tratamos com versão vetorizada
    eik_amplitudes = np.array([
        eik_amp_vec(sqrt_s, b_max, q_max, eps, mg, a1, a2, m2_func, n=n)
        for BA in born_amplitudes
    ])

    # Seção de choque diferencial (também vetorizada) 
    diff_sigma = differential_sigma(eik_amplitudes, s)

    return diff_sigma


In [24]:
def make_least_squares(x, y, yerr, energy, model_type):
    return LeastSquares(x, y, yerr, 
        lambda x, eps, mg, a1, a2: model_function(x, eps, mg, a1, a2, energy, model_type))

# Direct calculation without extra functions
total_cost_log_atlas = (
    make_least_squares(x_7_atlas, y_7_atlas, yerr_7_atlas, 7000, 'log') +
    make_least_squares(x_8_atlas, y_8_atlas, yerr_8_atlas, 8000, 'log') +
    make_least_squares(x_13_atlas, y_13_atlas, yerr_13_atlas, 13000, 'log')
)

total_cost_pl_atlas = (
    make_least_squares(x_7_atlas, y_7_atlas, yerr_7_atlas, 7000, 'pl') +
    make_least_squares(x_8_atlas, y_8_atlas, yerr_8_atlas, 8000, 'pl') +
    make_least_squares(x_13_atlas, y_13_atlas, yerr_13_atlas, 13000, 'pl')
)


In [25]:
def otimization(total_cost_func, initial_params, model_type: str, ensemble: str):
    print('\n')
    print(80 * '-')
    print(f"Iniciando otimização dos parâmetros usando LeastSquares para {model_type} em {ensemble.upper()}")

    start_time = time.time()
    
    m = Minuit(total_cost_func, 
               eps=initial_params['eps'],
               mg=initial_params['mg'],
               a1=initial_params['a1'],
               a2=initial_params['a2'])

    if model_type == 'pl':
        # Configurações adicionais
        down = 0.80
        up = 2 - down

        # m.limits['mg'] = (down * initial_params['mg'], up *initial_params['mg'])
        # m.limits['eps'] = (down * initial_params['eps'], up *initial_params['eps'])
        # m.limits['a2'] = (down * initial_params['a2'], up * initial_params['a2'])
        # m.limits['a1'] = (down * initial_params['a1'], up * initial_params['a1'])

        # Configurações adicionais
        m.strategy = 0
        m.errordef = 1
        m.tol = 1e-2

        ncall = 100


        # print('simplex...')
        # print(80*'-')
        # m.simplex(ncall=ncall)

        print('migrad 1 ...')
        print(80*'-')
        m.migrad(ncall=ncall)

        # print('migrad 2 ...')
        # print(80*'-')
        # m.migrad(ncall=ncall)

        # print('hesse...')
        # print(80*'-')
        # m.hesse(ncall=ncall)

        print('minos...')
        print(80*'-')
        m.minos(cl = 0.9)


    else:
        down = 0.94
        up = 2- down

        m.limits['mg'] = (down * initial_params['mg'], up * initial_params['mg'])
        # m.limits['eps'] = (down * initial_params['epsilon'], up * initial_params['epsilon'])
        m.limits['a2'] = (down * initial_params['a2'], up * initial_params['a2'])
        # m.limits['a1'] = (down * initial_params['a1'], up * initial_params['a1'])

        # Configurações adicionais
        m.strategy = 2
        m.errordef = 7.79
        m.tol = 1e-2
         
        m.migrad(ncall=70)
        m.hesse(ncall=10)

    print(f'Finalizado a minimização para {model_type} em {ensemble}')

    execution_time = time.time() - start_time
    
    minutes = int(execution_time // 60)
    seconds = execution_time % 60
    print(f'Tempo de execução para {model_type} em {ensemble}: {minutes} min {seconds:.2f} s \n')
    
    print(f"Parâmetros otimizados para {model_type} em {ensemble}: \n")
    print(f'mg: {m.values["mg"]} ± {m.errors["mg"]}')
    print(f'eps: {m.values["eps"]} ± {m.errors["eps"]}')
    print(f'a1: {m.values["a1"]} ± {m.errors["a1"]}')
    print(f'a2: {m.values["a2"]} ± {m.errors["a2"]}')
    print(f'chi2/ndof: {m.fval / m.ndof}')


    return m



In [26]:
m_pl_atlas = otimization(total_cost_pl_atlas,
                         initial_params_pl_atlas, 'pl', 'atlas')


/home/victorli/miniconda3/envs/two_gluon/lib/python3.11/site-packages/iminuit/minuit.py:139: ErrordefAlreadySetWarning: cost function has an errordef attribute equal to 1.0, you should not override this with Minuit.errordef
  warnings.warn(msg, ErrordefAlreadySetWarning)




--------------------------------------------------------------------------------
Iniciando otimização dos parâmetros usando LeastSquares para pl em ATLAS
migrad 1 ...
--------------------------------------------------------------------------------


KeyboardInterrupt: 